# Stage 1: Geospatial and Operational Inventory of Warehouses 

**Project:** Southern California Warehouse Inventory and Compliance Profiling - Caltrans 65A1345
**Institutions:** Institute of Transportation Studies - UC Davis (UCD) 
**Regulatory context:** South Coast Air Quality Management District (South Coast AQMD) 

---

## Overview

This notebook constructs a comprehensive **geospatial and operational inventory** of warehouses ≥ 50,000 ft² 

**Research Question:** What are the warehouses and distribution centers' location patterns over time and space?

---

## Methodology Overview

| Step | Description | Key Output |
|------|-------------|------------|
| 1 | Define study area (4-county ROI) | `ee.Geometry` ROI |
| 2 | Collect building footprints from 4 sources | Per-county GeoDataFrames |
| 3 | Classify buildings (tags + names + geometry) | `gdf_industrial` with labels |
| 4 | Land use filter (spatial join to LULC data) | `gdf_final` with LU class |
| 5 | Visualization and export | `.gpkg` inventory file |
| 6 | Truck trip data (operational intensity) | `.gpkg` truck metrics |
| 7 | Validation and integration | `.gpkg` data integration |


---

## Classification Label System

Every building in the output carries one of three labels:

| Label | Meaning | `needs_review` flag |
|-------|---------|-------------------|
| `confirmed` | Strong positive signal (OSM tag, Overture class, or known operator name) | `False` |
| `candidate` | Passes all exclusion filters but lacks a positive signal — review geometric shape metrics | `True` |
| `excluded` | Matched an exclusion rule (tag, subtype/class, name, or land use zone) | `False` |

Excluded buildings are **retained** in the output for full audit traceability.
The `confidence_note` column records exactly which rule triggered each label.

## Truck Metrics 

- South Coast 
- PeMS classification + WIM stations: vehicle class shares/weights where available (historical archive).
- SCAG High-Cube Warehouse trip-rate study (CEQA): published trip rates per 1,000 ft² for high-cube facilities; use with caution and cite assumptions until AWR comes.

---

## Study Area

- **Counties:** Imperial · Los Angeles · Orange · Riverside · San Bernardino · Ventura
- **Timeframe:** 2021 – 2023 (annual)
- **Size threshold:** ≥ 100,000 ft² (≈ 9,290 m²)
- **CRS (analysis):** EPSG:3310 — California Albers (metric); exported as EPSG:4326


## 0. Environment Setup

### Requirements

Install dependencies before running (or use the provided `environment.yml`):

```bash
conda env create -f environment.yml
conda activate geo
```

**Key package versions (tested):**

| Package | Version |
|---------|---------|
| Python | 3.11.9 |
| geopandas | 0.14.3 |
| pandas | 2.3.2 |
| numpy | 1.26.4 |
| shapely | 2.1.2 |
| osmnx | latest |
| overturemaps | latest |
| google-earth-engine | latest |
| geemap | latest |

### Google Earth Engine Authentication

A GEE account is required to access NAIP imagery. Run `ee.Authenticate()` once
per machine. After that, only `ee.Initialize()` is needed per session.
See: https://developers.google.com/earth-engine/guides/auth


In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import json
import re
import os
import time
import gzip
import shutil
import tempfile
import zipfile
import subprocess
from io import BytesIO
from pathlib import Path
from datetime import datetime

# ── Scientific stack ───────────────────────────────────────────────────────────
import numpy as np                          # numpy 1.26.4
import pandas as pd                         # pandas 2.3.2
import matplotlib.pyplot as plt            # matplotlib 3.10.6
import statsmodels.api as sm
from scipy import stats

# ── Geospatial stack ───────────────────────────────────────────────────────────
import geopandas as gdp                    # geopandas 0.14.3
import fiona                               # fiona 1.9.6
import pyproj                              # pyproj 3.7.2
import folium
from shapely.geometry import box
from shapely.ops import unary_union

# ── Networking ────────────────────────────────────────────────────────────────
import requests
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

# ── Data sources ──────────────────────────────────────────────────────────────
import osmnx as ox                         # OpenStreetMap
import overturemaps as om                  # Overture Maps
from pystac_client import Client           # Microsoft Planetary Computer
import planetary_computer

# ── Paths ─────────────────────────────────────────────────────────────────────
# TODO: Set this to your local project root before running.
# All relative paths in this notebook use this as the base.
# Example: path = '/home/username/project6/'
path = 'C:/Users/mariacvc/Box/Catalina Valencia/Project 6 - 2025ITS_RFP/Stage1/'


In [ ]:
# ── Google Earth Engine authentication & initialization ───────────────────────
# Only run ee.Authenticate() once per machine.
# On subsequent sessions, ee.Initialize() is sufficient.
import ee
import geemap

ee.Authenticate(auth_mode='notebook')   # Comment out after first run
ee.Initialize()
print("GEE initialized successfully.")


## 1: Define the Study Area

We use TIGER/2018 county boundaries accessed through Google Earth Engine to
define a Region of Interest (ROI) covering the core SCAG counties.
The ROI is used to spatially filter NAIP imagery in Step 2.

**Why these counties?**  
Los Angeles, Orange, Riverside, San Bernardino, Ventura, and Imperial 
form the regulatory jurisdiction of Souther California region


In [ ]:
def get_roi(county_names, state_fips='06', county_dataset='TIGER/2018/Counties'):
    """
    Build a single ee.Geometry covering one or more California counties.

    Queries the US Census TIGER/2018 county boundaries hosted on Google Earth
    Engine and unions all requested counties into one geometry. This ROI is
    used to clip NAIP imagery and filter building datasets by extent.

    Parameters
    ----------
    county_names : list of str
        County names exactly as they appear in TIGER data.
        Example: ['Los Angeles', 'Orange', 'Riverside', 'San Bernardino']
    state_fips : str
        State FIPS code. Default '06' = California.
    county_dataset : str
        GEE FeatureCollection asset path for county boundaries.

    Returns
    -------
    ee.Geometry
        Unioned geometry of all requested counties (EPSG:4326).

    Example
    -------
    >>> roi = get_roi(['Los Angeles', 'Orange'], state_fips='06')
    """
    counties = ee.FeatureCollection(county_dataset)

    # filter by state first
    state_counties = counties.filter(ee.Filter.eq('STATEFP', state_fips))

    # merge geometries for all selected counties
    roi = None
    for c in county_names:
        geom = state_counties.filter(ee.Filter.eq('NAME', c)).geometry()
        roi = geom if roi is None else roi.union(geom)

    return roi


## 2: Collect Primary Building Footprint Sources

We combine four complementary data sources to maximize coverage and
minimize classification gaps. No single source is sufficient alone:

| Dataset | Strength | Limitation |
|---------|----------|-----------|
| **USDA NAIP** | Detects all roof surfaces from aerial imagery | No semantic labels; requires post-processing |
| **Microsoft Buildings** | Highly accurate roof geometry; good for area estimation | No building type information |
| **Overture Maps** | Rich schema: `subtype`, `class`, operator names | Some coverage gaps in industrial parks |
| **OpenStreetMap** | Named facilities (e.g., "Amazon Fulfillment Center") | Sparse and inconsistent in many areas |

All four sources are:
- Legally permissible for research use
- Publicly accessible and reproducible
- Scientifically defensible for peer review
- Suitable for statewide scaling

**Size filter:** All sources are filtered to ≥ 50,000 ft².
Area is computed in EPSG:3310 (California Albers, metric) then converted to ft².


### 2.1 USDA NAIP — Roof Detection via Google Earth Engine

In [ ]:
def get_naip_image(roi, ini_date='2022-01-01', end_date='2022-12-31'):
    """
    Retrieve a cloud-free NAIP mosaic clipped to the ROI.

    NAIP (National Agriculture Imagery Program) provides ~0.6 m GSD aerial
    imagery with Red, Green, Blue, and Near-Infrared (RGBN) bands. The mosaic
    approach fills spatial gaps between individual flight strips.

    Parameters
    ----------
    roi       : ee.Geometry   Region of interest (output of get_roi())
    ini_date  : str           Start date in 'YYYY-MM-DD' format
    end_date  : str           End date in 'YYYY-MM-DD' format

    Returns
    -------
    ee.Image
        NAIP mosaic with bands R, G, B, N clipped to the ROI.
    """
    naip = (ee.ImageCollection('USDA/NAIP/DOQQ')
              .filterBounds(roi)
              .filterDate(ini_date, end_date)
              .mosaic()
              .clip(roi))
    return naip


def detect_large_roofs(roi, start_date='2022-01-01', end_date='2022-12-31',
                       area_threshold_m2=9290, scale=3):
    """
    Identify large impervious roof surfaces from NAIP imagery using spectral indices.

    Algorithm
    ---------
    1. Compute NDVI (vegetation index) and NDBI (built-up index) from NAIP bands.
    2. Mask pixels that are non-vegetated (NDVI < 0.4) and built-up (NDBI > 0).
    3. Remove small connected regions (< 20 pixels) to reduce noise.
    4. Vectorize the mask into polygon footprints.
    5. Filter polygons by minimum area (default ≈ 100,000 ft²).

    This produces candidate roof polygons that still need semantic classification
    in Step 3 — NAIP alone cannot distinguish warehouse roofs from parking lots
    or other large impervious surfaces.

    Parameters
    ----------
    roi               : ee.Geometry   Region of interest
    start_date        : str           NAIP start date ('YYYY-MM-DD')
    end_date          : str           NAIP end date ('YYYY-MM-DD')
    area_threshold_m2 : float         Minimum polygon area in m² (default 9290 ≈ 100k ft²)
    scale             : int           Pixel scale for vectorization in metres (default 3 m)

    Returns
    -------
    dict with keys:
        'naip'          : ee.Image             — original NAIP mosaic
        'roof_mask'     : ee.Image             — binary roof mask
        'roof_polygons' : ee.FeatureCollection — filtered roof polygons ≥ threshold
    """
    naip = get_naip_image(roi, start_date, end_date)

    # Spectral indices
    ndvi = naip.normalizedDifference(['N', 'R']).rename('NDVI')
    ndbi = naip.normalizedDifference(['G', 'N']).rename('NDBI')   # built-up proxy

    # Roof mask: impervious, non-vegetated
    roof_mask      = ndbi.gt(0.0).And(ndvi.lt(0.4)).selfMask()
    roof_connected = roof_mask.connectedPixelCount(50, True).gte(20).selfMask()

    # Vectorize
    vectors = roof_connected.reduceToVectors(
        geometry=roi, scale=scale, geometryType='polygon',
        eightConnected=True, labelProperty='roof',
        bestEffort=True, maxPixels=1e9
    )

    # Add area and filter
    vectors = vectors.map(lambda f: f.set('area_m2', ee.Geometry(f.geometry()).area()))
    roof_polygons = vectors.filter(ee.Filter.gt('area_m2', area_threshold_m2))

    count = roof_polygons.size().getInfo()
    print(f"NAIP roof polygons ≥ {area_threshold_m2:,} m² : {count:,}")

    return {'naip': naip, 'roof_mask': roof_connected, 'roof_polygons': roof_polygons}


### 2.2 Overture Maps — Building Footprints with Semantic Schema

Overture Maps consolidates building footprints from OpenStreetMap, Microsoft,
Google, and Esri into a unified schema. The key fields used here are:

- `subtype` — broad category (e.g., `industrial`, `commercial`, `residential`)
- `class`   — more specific type (e.g., `warehouse`, `logistics`, `hotel`)
- `names`   — primary and common names (JSON-formatted string)

**Download approach:** For large counties (LA, SB, RI), the bounding box is
split into tiles to avoid API timeouts. The tiles are then merged into a
single per-county GeoJSON.

**Overture API docs:** https://docs.overturemaps.org/guides/buildings/


In [ ]:
# ── Tiled download helpers ─────────────────────────────────────────────────────

def tile_bbox(bbox, nx=4, ny=4):
    """
    Subdivide a bounding box into an nx × ny grid of smaller tiles.

    Used to avoid timeout errors when downloading large counties from Overture.

    Parameters
    ----------
    bbox : list  [min_lon, min_lat, max_lon, max_lat]
    nx   : int   Number of columns (default 4)
    ny   : int   Number of rows (default 4)

    Returns
    -------
    list of [min_lon, min_lat, max_lon, max_lat] sub-tiles
    """
    minx, miny, maxx, maxy = bbox
    xs = np.linspace(minx, maxx, nx + 1)
    ys = np.linspace(miny, maxy, ny + 1)
    return [[xs[i], ys[j], xs[i+1], ys[j+1]] for i in range(nx) for j in range(ny)]


def download_overture_tiled(county_prefix, bbox, nx=4, ny=4):
    """
    Download Overture building footprints for a county using a tiled strategy.

    Splits the bounding box into nx × ny tiles and downloads each separately,
    sleeping 1 second between requests to respect rate limits.

    Parameters
    ----------
    county_prefix : str   Output filename prefix (e.g., 'la')
    bbox          : list  [min_lon, min_lat, max_lon, max_lat]
    nx, ny        : int   Grid dimensions for tiling

    Returns
    -------
    list of str   Paths to successfully downloaded tile GeoJSON files
    """
    tiles     = tile_bbox(bbox, nx, ny)
    out_files = []

    for k, tile in enumerate(tiles):
        bbox_str = ",".join(map(str, tile))
        out_file = f"{county_prefix}_tile{k}.geojson"
        cmd = ["overturemaps", "download", "--type=building",
               "--bbox", bbox_str, "-f", "geojson", "-o", out_file]

        print(f"⬇️  Tile {k}/{len(tiles)-1} → {out_file}")
        try:
            subprocess.run(cmd, check=True)
            out_files.append(out_file)
        except Exception as e:
            print(f"❌  Tile {k} failed: {e}")
        time.sleep(1)

    return out_files


def merge_overture_tiles(tile_files):
    """
    Merge a list of Overture tile GeoJSON files into one GeoDataFrame.

    Parameters
    ----------
    tile_files : list of str   Paths returned by download_overture_tiled()

    Returns
    -------
    GeoDataFrame or None if all tiles failed to load
    """
    gdfs = []
    for f in tile_files:
        try:
            gdfs.append(gdp.read_file(f))
        except Exception as e:
            print(f"  Skipping {f}: {e}")
    if not gdfs:
        print("❌  No tiles loaded.")
        return None
    merged = gdp.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=4326)
    print(f"✅  Merged {len(tile_files)} tiles → {len(merged):,} features")
    return merged


def run_overture_download(county_name, bbox):
    """
    Simple single-call Overture download (use for smaller counties).

    For large counties (LA, SB, RI) prefer download_overture_tiled() to avoid
    timeout errors from the Overture CLI.

    Parameters
    ----------
    county_name : str   Used as the output filename prefix
    bbox        : list  [min_lon, min_lat, max_lon, max_lat]
    """
    bbox_str = ",".join(map(str, bbox))
    out_file = f"{county_name.lower()}_buildings.geojson"
    cmd = ["overturemaps", "download", "--type=building",
           "--bbox", bbox_str, "-f", "geojson", "-o", out_file]
    print(f"⬇️  Downloading {county_name} buildings...")
    subprocess.run(cmd, check=True)
    print(f"✅  Saved → {out_file}")


# ── Load and filter ────────────────────────────────────────────────────────────

def get_overture_building(buildings_path, area_threshold_ft2=100_000, county=""):
    """
    Load a downloaded Overture GeoJSON and filter to large buildings only.

    Reprojects to EPSG:3310 (California Albers) to compute accurate planar
    areas, filters by the size threshold, then returns to EPSG:4326.

    Parameters
    ----------
    buildings_path     : str    Path to local Overture GeoJSON file
    area_threshold_ft2 : float  Minimum footprint area in ft² (default 100,000)
    county             : str    County label for progress reporting

    Returns
    -------
    GeoDataFrame
        Buildings ≥ threshold with 'area_m2' and 'area_ft2' columns added.

    Raises
    ------
    RuntimeError if the file cannot be read.
    """
    try:
        with fiona.Env():
            gdf = gdp.read_file(buildings_path)
    except Exception as e:
        raise RuntimeError(f"Failed to read {buildings_path}: {e}")

    gdf = gdf[gdf.geometry.notnull()].copy().to_crs(epsg=3310)
    gdf["area_m2"]  = gdf.geometry.area
    gdf["area_ft2"] = gdf["area_m2"] * 10.7639

    large = gdf[gdf["area_ft2"] >= area_threshold_ft2].copy().to_crs(epsg=4326)
    print(f"  {county}: {len(large):,} buildings ≥ {area_threshold_ft2:,} ft² (Overture)")
    return large


### 2.3 Microsoft US Building Footprints

Microsoft's building footprint dataset is derived from high-resolution satellite
imagery using computer vision. It provides excellent geometric accuracy (~0.5 m)
but contains no semantic building type information.

**Source:** https://github.com/microsoft/USBuildingFootprints  
**Download:** The full California file (~4–5 GB) is downloaded once and cached locally.
Subsequent runs use the cached ZIP, which is then spatially filtered to the county bbox.

**Note:** This is a large download. Ensure you have stable internet and ~6 GB of disk space.


In [ ]:
def get_ms_buildings(bbox, state_name="California", area_threshold_ft2=100_000,
                     cache_dir="data"):
    """
    Download Microsoft US Building Footprints and filter to a bounding box.

    Downloads the full state file once (cached), then reads only the buildings
    within the specified bounding box using spatial indexing. Filters to
    buildings ≥ area_threshold_ft2.

    Parameters
    ----------
    bbox               : list  [min_lon, min_lat, max_lon, max_lat] in EPSG:4326
    state_name         : str   State name as used in Microsoft's URL (default 'California')
    area_threshold_ft2 : float Minimum footprint area in ft² (default 100,000)
    cache_dir          : str   Directory to cache the downloaded ZIP (default 'data/')

    Returns
    -------
    GeoDataFrame
        Microsoft building footprints within bbox, ≥ threshold, in EPSG:4326.

    Notes
    -----
    Uses retry logic (5 attempts, exponential backoff) for robustness.
    The cached ZIP is reused on subsequent calls — delete it to force re-download.
    """
    os.makedirs(cache_dir, exist_ok=True)
    zip_path = os.path.join(cache_dir, f"{state_name}.geojson.zip")
    base_url = (f"https://minedbuildings.z5.web.core.windows.net/"
                f"legacy/usbuildings-v2/{state_name}.geojson.zip")

    # ── 1. Download (skip if cached) ──────────────────────────────────────────
    if not os.path.exists(zip_path):
        print(f"⬇️  Downloading {state_name} footprints (~4–5 GB) ...")
        session = requests.Session()
        retry   = Retry(total=5, backoff_factor=5,
                        status_forcelist=[500, 502, 503, 504],
                        allowed_methods=["GET"])
        session.mount("https://", HTTPAdapter(max_retries=retry))
        with session.get(base_url, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(zip_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=10_485_760):
                    if chunk:
                        f.write(chunk)
        print(f"✅  Saved → {zip_path}")
    else:
        print(f"✅  Using cached file: {zip_path}")

    # ── 2. Extract and read (bbox-filtered) ───────────────────────────────────
    with tempfile.TemporaryDirectory() as tmpdir:
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(tmpdir)
            geojson_files = [f for f in zf.namelist() if f.endswith(".geojson")]
            if not geojson_files:
                raise FileNotFoundError("No GeoJSON found inside ZIP.")
            geojson_path = os.path.join(tmpdir, geojson_files[0])

        print(f"📂  Reading subset for bbox {bbox} ...")
        gdf = gdp.read_file(geojson_path, bbox=bbox)

    # ── 3. Compute area and filter ────────────────────────────────────────────
    gdf = gdf.to_crs(3310)
    gdf["area_ft2"] = gdf.area * 10.7639
    gdf = gdf[gdf["area_ft2"] >= area_threshold_ft2].to_crs(4326)
    print(f"✅  {len(gdf):,} buildings ≥ {area_threshold_ft2:,} ft² (Microsoft)")
    return gdf


### 2.4 OpenStreetMap — Building Tags and Named Facilities

OSM provides the richest semantic information of the four sources. Named facilities
such as "Amazon Fulfillment Center" or "FedEx Ground" are often tagged in OSM
even when absent from other datasets.

We download both polygons (building footprints) and points (facility centroids).
Points are later buffered and used as semantic tag carriers to enrich Microsoft
and Overture footprints that overlap them (see Step 3).

**OSM tag reference:** https://wiki.openstreetmap.org/wiki/Key:building

**Important:** OSM reflects the state of the database at download time and may
not capture all facilities. Coverage is denser in urban areas and sparser in
industrial zones. Always cross-reference with Overture and Microsoft.


In [ ]:
ox.settings.timeout = 600   # 10-minute timeout for large county queries

def get_osm_buildings(place_name, download_year=None):
    """
    Download OSM building polygons and points for a named place.

    Queries the Overpass API via OSMnx for a broad set of building and
    amenity tags. Both polygons (footprints) and points (facility centroids)
    are returned. Points are used as semantic tag carriers in Step 3.

    Parameters
    ----------
    place_name    : str  Nominatim place name (e.g., 'Los Angeles County, California, USA')
    download_year : int  Year of download recorded in 'source_year' column.
                         Defaults to current year if not provided.

    Returns
    -------
    GeoDataFrame
        OSM features in EPSG:4326 with columns:
        ['geometry', 'name', 'building', 'tourism', 'amenity',
         'shop', 'healthcare', 'source_year']

    Notes
    -----
    Supports both OSMnx 1.x (geometries_from_place) and 2.x (features_from_place).
    Large counties (SB, Riverside) may take several minutes to download.
    """
    # Tags cover industrial/warehouse targets AND common non-industrial buildings
    # (non-industrial tags are needed so they can be used as exclusion signals in Step 3)
    tags = {
        'building':   ['industrial', 'warehouse', 'commercial',
                       'supermarket', 'retail', 'office', 'museum',
                       'school', 'university', 'sports_centre',
                       'stadium', 'military', 'hospital', 'church',
                       'cathedral', 'hotel', 'transportation',
                       'parking', 'apartments', 'residential'],
        'amenity':    ['cinema', 'casino', 'theatre', 'prison',
                       'police', 'ranger_station', 'fire_station',
                       'clinic', 'doctors', 'nursing_home',
                       'library', 'planetarium', 'arts_centre',
                       'community_centre', 'conference_centre',
                       'kindergarten', 'school', 'university',
                       'college', 'pharmacy', 'veterinary',
                       'social_facility', 'social_centre',
                       'shelter', 'marketplace'],
        'tourism':    ['zoo', 'aquarium', 'theme_park', 'hotel',
                       'motel', 'guest_house', 'chalet'],
        'leisure':    ['water_park', 'resort'],
        'shop':       ['supermarket', 'convenience', 'general',
                       'clothes', 'electronics', 'wholesale'],
        'healthcare': ['hospital', 'clinic', 'doctor',
                       'dentist', 'physiotherapist']
    }

    if download_year is None:
        download_year = datetime.now().year

    # OSMnx API compatibility: 1.x vs 2.x
    if hasattr(ox, "features_from_place"):
        gdf = ox.features_from_place(place_name, tags=tags)
    else:
        gdf = ox.geometries_from_place(place_name, tags=tags)

    gdf["source_year"] = download_year
    gdf = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon", "Point"])]

    columns_to_keep = ['geometry', 'name', 'building', 'tourism',
                       'amenity', 'shop', 'healthcare', 'source_year']
    gdf_reduced = gdf[[c for c in columns_to_keep if c in gdf.columns]].copy()

    print(f"  {place_name}: {len(gdf_reduced):,} OSM features downloaded")
    return gdf_reduced.to_crs(epsg=4326)


## Step 3: Building Classification

### Overview

Buildings from the three vector sources (OSM, Overture, Microsoft) are merged
and classified into one of three labels using a multi-signal rule system:

**Signal priority for Overture buildings:**
1. **Positive name match** — known logistics operator → `confirmed` (checked first so DCs are never accidentally excluded)
2. **Overture subtype/class exclusion** — e.g., `subtype=education` → `excluded`
3. **OSM tag exclusion** (from buffered point join) → `excluded`
4. **Name exclusion keyword** — e.g., "torrance memorial" → `excluded`
5. **Overture class confirmation** — e.g., `class=warehouse` → `confirmed`
6. **No signal** → `candidate` (reviewed using shape metrics)

**Microsoft buildings** have no semantic metadata, so only OSM tag exclusion applies.
All unexcluded MS buildings are initially `candidate` — shape metrics resolve these.

### Shape Metrics (for candidate resolution)

| Metric | Formula | Warehouse signal |
|--------|---------|-----------------|
| `compactness` | 4π·area / perimeter² | Low (<0.6) = irregular shape |
| `rectangularity` | area / MBR area | High (>0.8) = boxy warehouse |
| `elongation` | MBR long / short side | High (>2.0) = cross-dock layout |


In [ ]:
def overlap_ratio(row, ms_df):
    """
    Compute the Intersection over Union (IoU) between a building and its
    nearest Microsoft footprint match.

    Used as a quality metric when cross-referencing Overture polygons with
    Microsoft geometries — high IoU (> 0.6) indicates the same physical
    building was captured by both sources.

    Parameters
    ----------
    row   : Series   A row from an Overture GeoDataFrame after spatial join
    ms_df : GeoDataFrame   Microsoft buildings GeoDataFrame (index must be intact)

    Returns
    -------
    float   IoU score in [0, 1]. Returns 0 if no match or empty geometries.
    """
    idx = row.index_right
    if pd.isna(idx) or idx not in ms_df.index:
        return 0

    geom1 = row.geometry
    geom2 = ms_df.loc[idx].geometry

    if not geom1 or geom1.is_empty or not geom2 or geom2.is_empty:
        return 0
    if not geom1.intersects(geom2):
        return 0

    inter = geom1.intersection(geom2)
    union = geom1.union(geom2)
    if inter.is_empty or union.is_empty:
        return 0

    return inter.area / union.area if union.area > 0 else 0


### 3.1 Classification Lookup Tables

In [ ]:
# ==============================================================================
# OVERTURE SCHEMA LOOKUP TABLES
# These map Overture's subtype/class values to classification outcomes.
# Source: https://docs.overturemaps.org/schema/reference/buildings/building/
# ==============================================================================

# Overture subtypes that definitively indicate non-warehouse land use
EXCLUDE_SUBTYPES = {
    'education', 'residential', 'accommodation',
    'entertainment', 'civic', 'medical', 'religious'
}

# Overture class values that indicate non-warehouse use
EXCLUDE_CLASSES = {
    'hotel', 'school', 'hospital', 'apartments', 'parking',
    'retail', 'supermarket', 'office', 'church', 'government',
    'fire_station', 'police', 'stadium', 'museum', 'prison',
    'courthouse', 'library', 'kindergarten', 'college',
    'university', 'nursing_home', 'clinic', 'pharmacy',
    'veterinary', 'motel', 'guest_house', 'hostel',
    'sports_centre', 'water_park', 'theme_park', 'zoo',
    'aquarium', 'cinema', 'theatre', 'casino', 'shelter',
    'community_centre', 'arts_centre', 'marketplace'
}

# Overture class values that positively confirm industrial/warehouse use
CONFIRM_CLASSES = {
    'warehouse', 'industrial', 'logistics', 'storage',
    'manufacturing', 'data_center', 'utility', 'factory',
    'distribution', 'cold_storage', 'fulfillment'
}

# ==============================================================================
# OSM TAG EXCLUSION SET
# Used to exclude Microsoft/Overture buildings that spatially overlap
# OSM points or polygons with non-industrial tags.
# ==============================================================================
EXCLUDE_OSM_TAGS = {
    'commercial', 'supermarket', 'retail', 'office', 'museum',
    'school', 'university', 'sports_centre', 'stadium', 'military',
    'hospital', 'church', 'cathedral', 'hotel', 'transportation',
    'parking', 'apartments', 'residential', 'cinema', 'casino',
    'theatre', 'prison', 'police', 'ranger_station', 'fire_station',
    'clinic', 'doctors', 'nursing_home', 'zoo', 'aquarium',
    'theme_park', 'library', 'planetarium', 'arts_centre',
    'community_centre', 'conference_centre', 'kindergarten', 'college',
    'pharmacy', 'veterinary', 'social_facility', 'social_centre',
    'shelter', 'marketplace', 'motel', 'guest_house', 'chalet',
    'water_park', 'resort', 'convenience', 'general', 'clothes',
    'electronics', 'wholesale', 'dentist', 'physiotherapist',
    'highschool', 'place_of_worship', 'car_wash', 'townhall',
    'courthouse', 'embassy', 'government', 'public_building', 'car_repair'
}

# ==============================================================================
# NAME-BASED EXCLUSION KEYWORDS
# Applied to Overture 'names.primary' field via case-insensitive partial match.
# Checked AFTER positive inclusion keywords — so a building named
# "Target Distribution Center" is confirmed before these exclusion rules run.
# ==============================================================================
EXCLUDE_NAME_KEYWORDS = [
    # Healthcare
    'hospital', 'medical center', 'medical centre', 'health system',
    'kaiser permanente', 'dignity health', 'torrance memorial',
    'cedars-sinai', 'providence', 'urgent care', 'surgery center',
    'dialysis', 'rehabilitation center', 'convalescent',
    # Retail storefronts (NOT distribution centers)
    # NOTE: 'walmart dc', 'target dc', etc. are intentionally kept in INCLUDE list.
    #       Only storefront identifiers are listed here.
    'home depot', "lowe's", 'lowes', 'ikea', 'best buy',
    "sam's club", 'big lots', 'smart & final', 'grocery outlet',
    'costco wholesale', 'walmart supercenter', 'walmart neighborhood', 'target store',
    # Civic / Government
    'city hall', 'county courthouse', 'fire station', 'fire house',
    'police department', 'police station', 'sheriff', 'post office',
    'dmv ', 'department of motor', 'united states postal',
    'federal building', 'city of ', 'county of ',
    # Passenger transportation (not freight)
    'passenger terminal', 'airport terminal', 'cruise terminal',
    'amtrak station', 'metro station', 'transit center', 'bus depot',
    # Education
    'high school', 'middle school', 'elementary school',
    'unified school', 'school district', 'university of',
    'college of', 'community college', 'trade school',
    # Hotels
    'marriott', 'hilton', 'hyatt', 'sheraton', 'holiday inn',
    'hampton inn', 'courtyard by', 'residence inn', 'extended stay',
    'best western', 'la quinta',
    # Worship
    'church', 'temple', 'mosque', 'cathedral', 'faith community',
    'ministries', 'parish', 'synagogue', 'kingdom hall',
    # Entertainment
    'stadium', 'arena', 'convention center', 'convention centre',
    'event center', 'amphitheater', 'movie theater', 'cinemark',
    'amc theater', 'regal cinema',
    # Residential
    'apartment', 'senior living', 'assisted living', 'condominiums', 'townhomes',
]

# ==============================================================================
# NAME-BASED INCLUSION KEYWORDS
# Positive confirmation of warehouse / distribution / industrial use.
# Checked FIRST — a building matching any of these is immediately 'confirmed',
# even if it would otherwise be caught by an exclusion rule.
# This ensures Walmart DCs, Target DCs, etc. are never accidentally excluded.
# ==============================================================================
INCLUDE_NAME_KEYWORDS = [
    # Explicit function descriptors
    'fulfillment center', 'fulfillment centre',
    'distribution center', 'distribution centre',
    'distribution hub', 'sortation center', 'sort center',
    'returns center', 'delivery station',
    'cross-dock', 'crossdock', 'transload',
    'cold storage', 'refrigerated warehouse', 'frozen storage',
    # Major e-commerce / retail distribution centers (Rule 2305 targets)
    'amazon',                            # Amazon FC, DSP, Fresh DC
    'walmart dc', 'walmart distribution', 'walmart fulfillment',
    'target dc', 'target distribution', 'target fulfillment',
    'home depot distribution', 'home depot dc',
    'costco distribution', 'costco depot',
    'ikea distribution', "lowe's distribution", 'lowes distribution',
    # Parcel / freight carriers
    'fedex', 'ups ', 'united parcel service',
    'dhl', 'ontrac', 'lasership', 'usps distribution', 'usps processing',
    # 3PL and trucking operators
    'xpo logistics', 'ryder ', 'penske logistics',
    'schneider national', 'estes express', 'old dominion',
    'saia ', 'averitt', 'dayton freight', 'pilot freight',
    'kuehne', 'geodis', 'ceva logistics', 'db schenker',
    'nippon express', 'kintetsu', 'yusen logistics',
    # Industrial real estate / park names
    'prologis', 'industrial park', 'business park',
    'commerce park', 'trade center', 'industrial center',
    'logistics park', 'logistics center', 'intermodal',
    'corporate park', 'commerce center',
    # SoCal port / intermodal operators
    'maersk', 'trapac', 'everport', 'apm terminal', 'yang ming', 'evergreen logistics',
    # Food & beverage distribution (common in the Inland Empire)
    'niagara bottling', 'stater bros', 'sysco', 'us foods',
    'performance food', 'gordon food', 'shamrock foods',
    'reyes holdings', 'core-mark', 'mclane',
    # Manufacturing
    'manufacturing', 'fabrication', 'assembly plant',
    'bottling plant', 'packaging facility',
]


### 3.2 Helper Functions

In [ ]:
def parse_primary_name(name_field):
    """
    Extract the primary display name from Overture's nested names field.

    Overture stores names as a JSON object: {'primary': 'Name', 'common': null, ...}
    This field may arrive as a dict, a JSON string, a plain string, or None/NaN.

    Parameters
    ----------
    name_field : any   Raw value from the Overture 'names' column

    Returns
    -------
    str or None   The primary name string, or None if unavailable
    """
    if name_field is None:
        return None
    if isinstance(name_field, float) and np.isnan(name_field):
        return None
    if isinstance(name_field, dict):
        return name_field.get('primary', None)
    if isinstance(name_field, str):
        try:
            cleaned = name_field.replace("'", '"').replace('None', 'null')
            parsed  = json.loads(cleaned)
            return parsed.get('primary', None)
        except Exception:
            return name_field.strip()   # Return raw string as fallback
    return None


def name_matches(name_str, keyword_list):
    """
    Case-insensitive partial match of a name string against a keyword list.

    Parameters
    ----------
    name_str     : str        Building name to test
    keyword_list : list[str]  Keywords to match against

    Returns
    -------
    str or None   First matching keyword (truthy), or None if no match (falsy)

    Example
    -------
    >>> name_matches("Amazon Fulfillment Center - ONT8", INCLUDE_NAME_KEYWORDS)
    'amazon'
    """
    if not name_str:
        return None
    name_lower = name_str.lower()
    for kw in keyword_list:
        if kw in name_lower:
            return kw
    return None


def compute_shape_metrics(gdf):
    """
    Add geometric shape metrics to help identify warehouse-like buildings.

    Warehouses tend to be large, rectangular, and (for cross-docks) elongated.
    These metrics quantify those properties for use in candidate resolution.

    Metrics
    -------
    compactness   : 4π·area / perimeter²
                    Range [0, 1]. Perfect circle = 1.0.
                    Warehouses are typically 0.3–0.7.

    rectangularity : footprint area / minimum bounding rectangle area
                    Range [0, 1]. Perfect rectangle = 1.0.
                    Warehouses typically > 0.80.

    elongation    : MBR long side / MBR short side
                    Range ≥ 1. Cross-docks typically > 2.0.

    Parameters
    ----------
    gdf : GeoDataFrame   Buildings in any CRS (reprojected internally to EPSG:3310)

    Returns
    -------
    GeoDataFrame   Input with 'compactness', 'rectangularity', 'elongation' added.
                   Returned in EPSG:4326.
    """
    gdf = gdf.copy().to_crs(3310)
    gdf['area_m2']     = gdf.geometry.area
    gdf['perimeter_m'] = gdf.geometry.length
    gdf['compactness'] = (4 * np.pi * gdf['area_m2']) / (gdf['perimeter_m'] ** 2)

    def _mbr_metrics(geom):
        try:
            mbr      = geom.minimum_rotated_rectangle
            mbr_area = mbr.area
            coords   = list(mbr.exterior.coords)
            sides    = sorted([
                ((coords[i][0] - coords[i+1][0])**2 +
                 (coords[i][1] - coords[i+1][1])**2) ** 0.5
                for i in range(4)
            ])
            rect  = geom.area / mbr_area if mbr_area > 0 else np.nan
            elong = sides[-1] / sides[0]  if sides[0]  > 0 else np.nan
            return rect, elong
        except Exception:
            return np.nan, np.nan

    metrics               = gdf.geometry.apply(_mbr_metrics)
    gdf['rectangularity'] = metrics.apply(lambda x: x[0])
    gdf['elongation']     = metrics.apply(lambda x: x[1])

    return gdf.to_crs(4326)


def _is_excluded_by_osm_tags(row, tag_columns):
    """
    Check whether any OSM semantic tag column in a row contains an exclusion value.

    Parameters
    ----------
    row         : Series      A row from a building GeoDataFrame
    tag_columns : list[str]   OSM tag column names to check (e.g., ['building', 'amenity'])

    Returns
    -------
    bool   True if the row should be excluded based on its OSM tags
    """
    for col in tag_columns:
        val = row.get(col)
        if isinstance(val, str) and val in EXCLUDE_OSM_TAGS:
            return True
    return False


### 3.3 Main Classifier

In [ ]:
def industrial_buildings(osm, ov, ms, buffer_m=20):
    """
    Classify warehouse and industrial buildings from three geospatial sources.

    Combines OSM, Overture, and Microsoft footprints and applies a multi-signal
    rule system to assign each building one of three classification labels.

    Classification Labels
    ---------------------
    'confirmed'  : High-confidence warehouse/industrial. Triggered by at least
                   one strong positive signal:
                   - OSM building tag = 'industrial' or 'warehouse'
                   - Overture class in CONFIRM_CLASSES
                   - Name matches INCLUDE_NAME_KEYWORDS
                   - Land use zone = Industrial or Transport (Step 4)

    'candidate'  : Passes all exclusion filters but has no positive signal.
                   Geometric shape metrics (rectangularity, elongation) should
                   be reviewed to accept or reject these. Flag: needs_review=True.

    'excluded'   : Matched at least one exclusion signal. Retained in output
                   with label and confidence_note for audit purposes.

    Parameters
    ----------
    osm      : GeoDataFrame   OSM features (polygons + points) in EPSG:4326
    ov       : GeoDataFrame   Overture buildings in EPSG:4326
    ms       : GeoDataFrame   Microsoft building footprints in EPSG:4326
    buffer_m : int            Buffer radius in metres for OSM point tag carriers.
                              OSM facility points are buffered so they can enrich
                              overlapping Overture and Microsoft polygons.

    Returns
    -------
    GeoDataFrame with columns:
        source           : 'osm' | 'overture' | 'ms'
        label            : 'confirmed' | 'candidate' | 'excluded'
        confidence_note  : Rule that triggered the label (e.g., 'overture_class:warehouse')
        primary_name     : Parsed building name (from Overture or OSM)
        needs_review     : True if label == 'candidate'
        compactness      : Shape metric (see compute_shape_metrics)
        rectangularity   : Shape metric
        elongation       : Shape metric

    Notes
    -----
    Naming conflict warning: Do NOT assign the output to a variable named
    'industrial_buildings' — this would overwrite the function reference.
    Use a distinct name such as 'gdf_industrial' instead.
    """

    # ── 1. Split OSM into polygons (footprints) and points (tag carriers) ─────
    osm_poly = osm[osm.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    osm_pts  = osm[osm.geom_type == "Point"].copy()

    # ── 2. Confirmed OSM industrial polygons ──────────────────────────────────
    osm_confirmed = osm_poly[osm_poly['building'].isin(['industrial', 'warehouse'])].copy()
    osm_confirmed['label']           = 'confirmed'
    osm_confirmed['confidence_note'] = 'osm_building_tag'
    osm_confirmed['source']          = 'osm'
    osm_confirmed['primary_name']    = osm_confirmed.get('name', None)

    # ── 3. Buffer OSM points → tag carriers ───────────────────────────────────
    # Buffering turns point locations into small polygons so they can be
    # spatially joined to overlapping Overture/Microsoft footprints.
    if len(osm_pts) > 0:
        osm_pts = osm_pts.to_crs(3310)
        osm_pts["geometry"] = osm_pts.geometry.buffer(buffer_m)
        osm_pts = osm_pts.to_crs(4326)

    tag_columns   = ['building', 'amenity', 'tourism', 'leisure', 'shop', 'healthcare']
    existing_cols = [c for c in tag_columns if c in osm_pts.columns] if len(osm_pts) > 0 else []

    # ── 4. Enrich Overture with OSM point tags ────────────────────────────────
    ov_work = ov.copy()
    if len(osm_pts) > 0 and existing_cols:
        ov_proj      = ov_work.to_crs(3310)
        osm_pts_proj = osm_pts.to_crs(3310)
        ov_join      = gdp.sjoin(
            ov_proj,
            osm_pts_proj[existing_cols + ["geometry"]],
            how="left", predicate="intersects"
        ).drop(columns=["geometry_right"], errors="ignore")
        ov_work = ov_join.to_crs(4326)

    # ── 5. Parse Overture name field ──────────────────────────────────────────
    ov_work['primary_name'] = ov_work['names'].apply(parse_primary_name)

    # ── 6. Classify each Overture building ────────────────────────────────────
    def _classify_overture(row):
        """Apply multi-signal classification to a single Overture row."""
        subtype = row.get('subtype') or ''
        cls     = row.get('class')   or ''
        name    = row.get('primary_name') or ''

        # Step 1: Positive name keyword wins — prevents DCs from being excluded
        pos_kw = name_matches(name, INCLUDE_NAME_KEYWORDS)
        if pos_kw:
            return 'confirmed', f'name_keyword:{pos_kw}'
        # Step 2: Overture schema exclusion
        if subtype in EXCLUDE_SUBTYPES:
            return 'excluded', f'overture_subtype:{subtype}'
        if cls in EXCLUDE_CLASSES:
            return 'excluded', f'overture_class:{cls}'
        # Step 3: OSM tag exclusion (from buffered point spatial join)
        if _is_excluded_by_osm_tags(row, existing_cols):
            hit = next((row.get(c) for c in existing_cols
                        if isinstance(row.get(c), str) and row.get(c) in EXCLUDE_OSM_TAGS),
                       'unknown')
            return 'excluded', f'osm_tag:{hit}'
        # Step 4: Name exclusion keyword
        neg_kw = name_matches(name, EXCLUDE_NAME_KEYWORDS)
        if neg_kw:
            return 'excluded', f'name_keyword_exclude:{neg_kw}'
        # Step 5: Confirmed Overture class
        if cls in CONFIRM_CLASSES:
            return 'confirmed', f'overture_class:{cls}'
        # Step 6: No signal — flag for shape metric review
        return 'candidate', 'no_signal'

    results = ov_work.apply(_classify_overture, axis=1, result_type='expand')
    results.columns = ['label', 'confidence_note']
    ov_work = pd.concat([ov_work, results], axis=1)
    ov_work['source'] = 'overture'

    # ── 7. Classify Microsoft footprints ──────────────────────────────────────
    # MS has no native semantic metadata — only OSM tag exclusion applies.
    # Unexcluded MS buildings become 'candidate' and rely on shape metrics.
    ms_work = ms.copy()
    if len(osm_pts) > 0 and existing_cols:
        ms_proj = ms_work.to_crs(3310)
        ms_join = gdp.sjoin(
            ms_proj,
            osm_pts_proj[existing_cols + ["geometry"]],
            how="left", predicate="intersects"
        ).drop(columns=["geometry_right"], errors="ignore")
        ms_work = ms_join.to_crs(4326)

    ms_work['primary_name'] = None

    def _classify_ms(row):
        if _is_excluded_by_osm_tags(row, existing_cols):
            hit = next((row.get(c) for c in existing_cols
                        if isinstance(row.get(c), str) and row.get(c) in EXCLUDE_OSM_TAGS),
                       'unknown')
            return 'excluded', f'osm_tag:{hit}'
        return 'candidate', 'ms_no_semantic_tag'

    ms_results = ms_work.apply(_classify_ms, axis=1, result_type='expand')
    ms_results.columns = ['label', 'confidence_note']
    ms_work = pd.concat([ms_work, ms_results], axis=1)
    ms_work['source'] = 'ms'

    # ── 8. Merge all sources; retain excluded for audit trail ─────────────────
    final = pd.concat(
        [osm_confirmed, ov_work, ms_work], ignore_index=True
    ).drop_duplicates(subset='geometry')

    # ── 9. Compute shape metrics on all buildings ─────────────────────────────
    final = compute_shape_metrics(final)

    # ── 10. Convenience flag for QGIS / geemap filtering ─────────────────────
    final['needs_review'] = final['label'] == 'candidate'

    # ── 11. Summary ───────────────────────────────────────────────────────────
    print("=" * 55)
    print("  Industrial Buildings Classifier — Summary")
    print("=" * 55)
    for lbl, cnt in final['label'].value_counts().items():
        print(f"  {lbl:<12} : {cnt:>6,}")
    print(f"  {'TOTAL':<12} : {len(final):>6,}")
    print("-" * 55)
    print("  Top confidence notes (candidates):")
    for note, cnt in final[final['label'] == 'candidate']['confidence_note'].value_counts().head(5).items():
        print(f"    {note:<35} : {cnt:>5,}")
    print("=" * 55)

    return final


## Step 4: Land Use Filter

### Overview

After semantic classification (Step 3), we apply a spatial land use filter
using OSM-derived LULC (Land Use / Land Cover) training shapefiles.
This adds a parcel-level spatial confirmation layer to the classification.

### Land Use Classes

| Class | Treatment | Rationale |
|-------|-----------|-----------|
| `Industrial` | **Confirm** (upgrades candidate) | Direct positive signal |
| `Transport` | **Confirm** (upgrades candidate) | Port/freight lands; WAIRE-relevant |
| `Residential` | **Exclude** (unless already confirmed) | No warehouses expected |
| `Commercial` | **Exclude** (unless already confirmed) | Storefronts, not DCs |
| `Institutional` | **Exclude** (unless already confirmed) | Hospitals, schools, civic |
| `Agriculture` | Neutral — no change | Cold storage warehouses possible |
| `Bare_land/other` | Neutral — no change | May be new development |
| `Nature_Vegetation` | Neutral — no change | Unlikely but not ruled out |
| `Water/Wetlands` | Neutral — no change | False positive protection |
| No LU match | Keep as candidate | No evidence either way |

### Conflict Rule

A `confirmed` building (e.g., named "Amazon Fulfillment Center") sitting inside
a `Residential` or `Commercial` zone is **kept** with a `lu_conflict_kept` flag
in `confidence_note`. 

### County Coverage

| County | File |
|--------|------|
| Los Angeles | `LA_osm_lulc_training.shp` |
| Orange | `OR_osm_lulc_training.shp` |
| Riverside | `RI_osm_lulc_training.shp` |
| San Bernardino | `SB_osm_lulc_training.shp` |
| Ventura | `VE_osm_lulc_training.shp` |
| Imperial | `IM_osm_lulc_training.shp` |


In [ ]:
# ==============================================================================
# LAND USE CLASS LOOKUP TABLES
# Values must match the 'landuse_gr' column in your county shapefiles exactly.
# ==============================================================================

LU_EXCLUDE_CLASSES = {
    'Residential',    # Single/multi-family housing
    'Commercial',     # Retail, strip malls — not WAIRE-regulated
    'Institutional',  # Schools, hospitals, civic buildings
}

LU_CONFIRM_CLASSES = {
    'Industrial',
    'Transport',      # Freight terminals, rail yards, port lands
}

LU_NEUTRAL_CLASSES = {
    'Agriculture',
    'Bare_land/other',
    'Nature_Vegetation',
    'Water/Wetlands',
}


def load_lu_layers(lu_paths, lu_column='landuse_gr', target_crs=4326):
    """
    Load and merge county-level land use shapefiles into a single GeoDataFrame.

    Validates that the expected column exists in each file, reprojects to a
    common CRS, and warns about any unexpected class values.

    Parameters
    ----------
    lu_paths   : dict  {county_label: path_to_shapefile}
                 Example: {'LA': 'data/LA_osm_lulc_training.shp', ...}
    lu_column  : str   Column containing land use class values (default 'landuse_gr')
    target_crs : int   EPSG code for output CRS (default 4326)

    Returns
    -------
    GeoDataFrame
        Merged LU layer with columns [lu_column, 'county', 'geometry']

    Raises
    ------
    ValueError     if lu_column is missing from any shapefile
    RuntimeError   if no shapefiles could be loaded
    """
    frames = []

    for county, path in lu_paths.items():
        p = Path(path)
        if not p.exists():
            print(f"  WARNING: Not found — {county}: {p} (skipping)")
            continue

        gdf = gdp.read_file(p)

        if lu_column not in gdf.columns:
            raise ValueError(
                f"Column '{lu_column}' not found in {p.name}. "
                f"Available: {gdf.columns.tolist()}"
            )

        gdf = gdf[[lu_column, 'geometry']].copy()
        gdf['county'] = county

        if gdf.crs is None:
            print(f"  WARNING: {county} has no CRS — assuming EPSG:4326")
            gdf = gdf.set_crs(4326)
        elif gdf.crs.to_epsg() != target_crs:
            gdf = gdf.to_crs(target_crs)

        unexpected = set(gdf[lu_column].dropna().unique()) - (
            LU_EXCLUDE_CLASSES | LU_CONFIRM_CLASSES | LU_NEUTRAL_CLASSES
        )
        if unexpected:
            print(f"  NOTE: {county} has unexpected values {unexpected} — treated as neutral")

        frames.append(gdf)
        print(f"  Loaded {county}: {len(gdf):,} LU polygons")

    if not frames:
        raise RuntimeError("No LU shapefiles loaded. Check your lu_paths.")

    merged = gdp.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=target_crs)
    print(f"\n  Total LU polygons: {len(merged):,}")
    print(merged[lu_column].value_counts().to_string())
    return merged


def assign_lu_to_buildings(buildings_gdf, lu_gdf, lu_column='landuse_gr',
                            use_centroid=True):
    """
    Spatially join land use class to each building footprint.

    Uses a two-pass strategy:
    1. Primary:  centroid-within join (fast, produces clean one-to-one matches)
    2. Fallback: largest-overlap join for buildings whose centroid falls outside
                 all LU polygons (common at county boundary edges)

    When a building centroid falls in multiple LU polygons (boundary overlap),
    the 'industrial wins' rule prioritises confirm classes over neutral classes
    over exclude classes.

    Parameters
    ----------
    buildings_gdf : GeoDataFrame   Output of industrial_buildings() or apply_lu_filter()
    lu_gdf        : GeoDataFrame   Merged LU layer from load_lu_layers()
    lu_column     : str            LU class column name
    use_centroid  : bool           Use centroid strategy (default True)

    Returns
    -------
    GeoDataFrame   Input buildings with 'lu_class' column added.
                   NaN = no LU polygon matched (treated as neutral).
    """
    work    = buildings_gdf.copy().to_crs(3310)
    lu_proj = lu_gdf.to_crs(3310)

    # ── Primary join: centroid within LU polygon ──────────────────────────────
    centroids            = work.copy()
    centroids['geometry'] = work.geometry.centroid

    joined = gdp.sjoin(
        centroids[['geometry']],
        lu_proj[[lu_column, 'geometry']],
        how='left', predicate='within'
    )

    # Resolve duplicates: industrial wins
    if joined.index.duplicated().any():
        priority_order    = list(LU_CONFIRM_CLASSES) + list(LU_NEUTRAL_CLASSES) + list(LU_EXCLUDE_CLASSES)
        joined[lu_column] = pd.Categorical(joined[lu_column], categories=priority_order, ordered=True)
        joined            = joined.sort_values(lu_column).loc[~joined.index.duplicated(keep='first')]

    work['lu_class'] = joined[lu_column]

    # ── Fallback: largest overlap for unmatched buildings ─────────────────────
    unmatched_idx = work[work['lu_class'].isna()].index
    if len(unmatched_idx) > 0:
        print(f"  Fallback overlap join for {len(unmatched_idx):,} unmatched buildings ...")
        unmatched    = work.loc[unmatched_idx].copy()
        overlap_join = gdp.sjoin(
            unmatched[['geometry']], lu_proj[[lu_column, 'geometry']],
            how='left', predicate='intersects'
        ).reset_index().rename(columns={'index': 'bldg_idx'})

        if not overlap_join.empty:
            def _max_overlap_class(group):
                bldg_geom  = work.loc[group['bldg_idx'].iloc[0], 'geometry']
                best_class = None
                best_area  = 0
                for _, row in group.iterrows():
                    lu_idx = row.get('index_right')
                    if pd.isna(lu_idx):
                        continue
                    try:
                        inter = bldg_geom.intersection(lu_proj.iloc[int(lu_idx)].geometry).area
                        if inter > best_area:
                            best_area  = inter
                            best_class = row[lu_column]
                    except Exception:
                        continue
                return best_class

            fallback = overlap_join.groupby('bldg_idx').apply(_max_overlap_class)
            work.loc[fallback.index, 'lu_class'] = fallback.values

    return work.to_crs(4326)


def apply_lu_filter(buildings_gdf, lu_gdf, lu_column='landuse_gr'):
    """
    Apply the land use filter to update building classification labels.

    Calls assign_lu_to_buildings() then applies the classification rules:
    - Industrial / Transport zone  → upgrade candidate to confirmed
    - Residential / Commercial / Institutional zone → exclude (unless confirmed)
    - Neutral zone or no match     → no change
    - Conflict (confirmed + exclusion zone) → keep with lu_conflict_kept flag

    Parameters
    ----------
    buildings_gdf : GeoDataFrame   Output of industrial_buildings()
    lu_gdf        : GeoDataFrame   Merged LU layer from load_lu_layers()
    lu_column     : str            LU class column name (default 'landuse_gr')

    Returns
    -------
    GeoDataFrame
        Updated buildings with 'lu_class', 'label', 'confidence_note',
        and 'needs_review' columns updated.
    """
    print("=" * 55)
    print("  Land Use Filter")
    print("=" * 55)

    work         = assign_lu_to_buildings(buildings_gdf, lu_gdf, lu_column)
    label_before = work['label'].copy()

    def _apply_lu_rule(row):
        lu    = row.get('lu_class')
        label = row.get('label', 'candidate')
        note  = row.get('confidence_note', '')

        if pd.isna(lu) or lu not in (LU_EXCLUDE_CLASSES | LU_CONFIRM_CLASSES):
            return label, note                                  # neutral / no match

        if lu in LU_CONFIRM_CLASSES:
            if label == 'candidate':
                return 'confirmed', f'lu_confirm:{lu}'
            return label, f'{note}|lu_confirm:{lu}'            # already confirmed

        if lu in LU_EXCLUDE_CLASSES:
            if label == 'confirmed':
                return label, f'{note}|lu_conflict_kept:{lu}'  # keep, flag conflict
            return 'excluded', f'lu_exclude:{lu}'

        return label, note

    results = work.apply(_apply_lu_rule, axis=1, result_type='expand')
    results.columns   = ['label', 'confidence_note']
    work['label']     = results['label']
    work['confidence_note'] = results['confidence_note']
    work['needs_review']    = work['label'] == 'candidate'

    # ── Summary ───────────────────────────────────────────────────────────────
    changed = (label_before != work['label']).sum()
    print(f"\n  Labels changed: {changed:,}")
    print("\n  Final label distribution:")
    for lbl, cnt in work['label'].value_counts().items():
        print(f"    {lbl:<12} : {cnt:>6,}")
    print("\n  LU class distribution:")
    for lu_cls, cnt in work['lu_class'].value_counts(dropna=False).items():
        print(f"    {str(lu_cls):<25} : {cnt:>6,}")

    conflicts = work[work['confidence_note'].str.contains('lu_conflict_kept', na=False)]
    if len(conflicts) > 0:
        print(f"\n  AUDIT: {len(conflicts):,} confirmed buildings in exclusion zones.")
        print("  Filter in QGIS: confidence_note LIKE '%lu_conflict_kept%'")
    print("=" * 55)
    return work


## Results: Running the Full Pipeline

This section executes the complete pipeline from data download through export.

### Before Running

1. Set `path` in the Setup cell to your local project directory.
2. Ensure Overture GeoJSON files have been downloaded to the working directory
   (either via the download functions above or the `overturemaps` CLI).
3. Ensure county LU shapefiles are available in `path + 'Shapefiles/'`.

### County Bounding Boxes

| County | min_lon | min_lat | max_lon | max_lat |
|--------|---------|---------|---------|---------|
| Los Angeles | -118.9448 | 33.7036 | -117.6463 | 34.8233 |
| Orange | -117.945 | 33.385 | -117.412 | 33.927 |
| Riverside | -117.705 | 33.434 | -114.434 | 34.119 |
| San Bernardino | -118.151 | 33.503 | -114.131 | 35.815 |
| Ventura | -119.75 | 33.00 | -118.63 | 34.90 |
| Imperial | # | # | # | # |

### Expected Runtimes (approximate)

| Step | Time |
|------|------|
| Overture download (per county) | 5–30 min |
| Microsoft download (California) | 20–60 min (one-time) |
| OSM download (per county) | 2–10 min |
| Classification (all counties) | 5–15 min |
| LU filter | 5–20 min |


In [ ]:
# ── County bounding boxes (EPSG:4326) ─────────────────────────────────────────
bboxes = {
    "la": (-118.9448, 33.7036, -117.6463, 34.8233),
    "or": (-117.945,  33.385,  -117.412,  33.927),
    "ri": (-117.705,  33.434,  -114.434,  34.119),
    "sb": (-118.151,  33.503,  -114.131,  35.815),
    "ve": (-119.75,   33.00,   -118.63,   34.90),
}

# ── Download Overture data (uncomment to run) ─────────────────────────────────
# For large counties, use the tiled downloader to avoid API timeouts.
# Run once; outputs are cached as .geojson files in the working directory.
#
# for county, bbox in bboxes.items():
#     run_overture_download(county, bbox)
#     time.sleep(5)
#
# Single county example:
# ! overturemaps download --bbox=-118.9448,33.7036,-117.6463,34.8233 -f geojson --type=building -o la_buildings.geojson


In [ ]:
# ==============================================================================
# STEP 1: Study Area
# ==============================================================================
sca_counties = ['Los Angeles', 'Orange', 'Riverside', 'San Bernardino']
roi = get_roi(sca_counties, state_fips='06')
print("ROI defined for:", sca_counties)

# ==============================================================================
# STEP 2: Load Building Footprints
# ==============================================================================

# ── NAIP (Google Earth Engine) ────────────────────────────────────────────────
naip_img      = get_naip_image(roi, '2022-01-01', '2022-12-31')
naip_buildings = detect_large_roofs(roi, area_threshold_m2=9_000)

# ── Overture Maps ─────────────────────────────────────────────────────────────
la_overture = get_overture_building("la_buildings.geojson", county="LA")
or_overture = get_overture_building("or_buildings.geojson", county="OR")
ri_overture = get_overture_building("ri_buildings.geojson", county="RI")
sb_overture = get_overture_building("sb_buildings.geojson", county="SB")
ve_overture = get_overture_building("ve_buildings.geojson", county="VE")

# ── Microsoft Building Footprints ─────────────────────────────────────────────
la_ms = get_ms_buildings(bboxes["la"])
or_ms = get_ms_buildings(bboxes["or"])
ri_ms = get_ms_buildings(bboxes["ri"])
sb_ms = get_ms_buildings(bboxes["sb"])
ve_ms = get_ms_buildings(bboxes["ve"])

# ── OpenStreetMap ─────────────────────────────────────────────────────────────
la_osm = get_osm_buildings("Los Angeles County, California, USA")
or_osm = get_osm_buildings("Orange County, California, USA")
ri_osm = get_osm_buildings("Riverside County, California, USA")
sb_osm = get_osm_buildings("San Bernardino County, California, USA")
ve_osm = get_osm_buildings("Ventura County, California, USA")

# ==============================================================================
# STEP 3: Merge Sources and Classify
# ==============================================================================

# Merge all counties per source
ov_buildings = gdp.GeoDataFrame(
    pd.concat([la_overture, or_overture, ri_overture, sb_overture, ve_overture],
              ignore_index=True), crs=4326)

ms_buildings = gdp.GeoDataFrame(
    pd.concat([la_ms, or_ms, ri_ms, sb_ms, ve_ms],
              ignore_index=True), crs=4326)

osm_buildings = gdp.GeoDataFrame(
    pd.concat([la_osm, or_osm, ri_osm, sb_osm, ve_osm],
              ignore_index=True), crs=4326)

# Reduce OSM to classifier-required columns
osm_cols   = ['geometry', 'name', 'building', 'tourism', 'amenity', 'source_year']
osm_reduced = osm_buildings[[c for c in osm_cols if c in osm_buildings.columns]].copy()

# Run classifier — NOTE: output variable name must differ from function name
gdf_industrial = industrial_buildings(osm_reduced, ov_buildings, ms_buildings)

# ==============================================================================
# STEP 4: Land Use Filter
# ==============================================================================

# Define paths to county LU shapefiles
# TODO: Update paths to match your local directory structure
lu_paths = {
    'LA': path + 'Shapefiles/LA_osm_lulc_training.shp',
    'OR': path + 'Shapefiles/OR_osm_lulc_training.shp',
    'RI': path + 'Shapefiles/RI_osm_lulc_training.shp',
    'SB': path + 'Shapefiles/SB_osm_lulc_training.shp',
    'VE': path + 'Shapefiles/VE_osm_lulc_training.shp',
    # Add Imperial and San Joaquin when ready:
    # 'IMP': path + 'Shapefiles/IMP_osm_lulc_training.shp',
    # 'SJ':  path + 'Shapefiles/SJ_osm_lulc_training.shp',
}

lu_merged = load_lu_layers(lu_paths)
gdf_final = apply_lu_filter(gdf_industrial, lu_merged)

# ==============================================================================
# EXPORT
# ==============================================================================
output_path = path + "warehouse_inventory_lu_filtered.gpkg"
gdf_final.to_file(output_path, driver="GPKG")
print(f"\n✅  Inventory exported → {output_path}")
print(f"    Total buildings: {len(gdf_final):,}")
print(f"    Confirmed:       {(gdf_final['label']=='confirmed').sum():,}")
print(f"    Candidates:      {(gdf_final['label']=='candidate').sum():,}")
print(f"    Excluded:        {(gdf_final['label']=='excluded').sum():,}")


## Visualization

Two interactive maps are provided:

1. **Source comparison** — shows raw Microsoft and Overture footprints side-by-side (sampled to 5,000 buildings each for performance)
2. **NAIP + classification** — overlays NAIP imagery with detected roof polygons and the classified building inventory

Use the layer toggles in geemap to compare sources. For detailed QA, open the exported `.gpkg` in QGIS and filter by `label`, `confidence_note`, or `needs_review`.

### QGIS Quick-Start Filters

| Filter | Expression |
|--------|-----------|
| Confirmed warehouses | `"label" = 'confirmed'` |
| Candidates needing review | `"needs_review" = 1` |
| Excluded buildings | `"label" = 'excluded'` |
| Confirmed DCs in exclusion zones | `"confidence_note" LIKE '%lu_conflict_kept%'` |
| Overture-confirmed warehouses | `"confidence_note" LIKE 'overture_class%'` |


In [ ]:
# ── Map 1: Source comparison ──────────────────────────────────────────────────
Map1 = geemap.Map(center=[34.05, -118.25], zoom=8)

Map1.add_gdf(
    ms_buildings.sample(min(5000, len(ms_buildings))),
    layer_name="Microsoft Buildings (≥100k ft²)",
    style={"color": "orange", "fillColor": "none", "weight": 1}
)
Map1.add_gdf(
    ov_buildings.sample(min(5000, len(ov_buildings))),
    layer_name="Overture Buildings (≥100k ft²)",
    style={"color": "blue", "fillColor": "none", "weight": 1}
)
Map1.add_gdf(
    gdf_final[gdf_final['label'] == 'confirmed'],
    layer_name="Confirmed Warehouses",
    style={"color": "red", "fillColor": "#ff000033", "weight": 1}
)

Map1


In [ ]:
# ── Map 2: NAIP imagery + classified inventory ────────────────────────────────
Map2 = geemap.Map(center=[34.05, -118.25], zoom=8)

try:
    Map2.addLayer(naip_buildings['naip'],
                  {'bands': ['R', 'G', 'B'], 'min': 0, 'max': 255}, 'NAIP 2022')
    Map2.addLayer(naip_buildings['roof_mask'],
                  {'palette': ['yellow']}, 'Roof Mask')
    Map2.addLayer(
        naip_buildings['roof_polygons'].style(color='orange', fillColor='00000000', width=1),
        {}, 'Large Roof Polygons (NAIP)'
    )
except NameError:
    print("NAIP layers not available — run detect_large_roofs() first.")

try:
    Map2.addLayer(roi, {}, 'Study Area (ROI)')
except NameError:
    pass

Map2.add_gdf(
    gdf_final[gdf_final['label'] == 'confirmed'],
    layer_name="Confirmed Warehouses",
    style={"color": "red", "fillColor": "#ff000033", "weight": 1}
)
Map2.add_gdf(
    gdf_final[gdf_final['needs_review']],
    layer_name="Candidates (needs review)",
    style={"color": "purple", "fillColor": "none", "weight": 1}
)

Map2
